# Forensic по merchants для проблемного ИНН

Тетрадка содержит 3 блока:
1. Полная выгрузка по `ods_alpha.scd1_merchants` для выбранного ИНН (с привязкой к компаниям и договорам).
2. Быстрая сводка по количеству ТСП и качеству данных.
3. Разрез `c_nmrc` + связь с договорами и транзакциями за месяц.

In [ ]:
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
target_inn = '010500295503'  # <-- замени на проблемный ИНН
month_start = '2026-02-01'
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

print('target_inn =', target_inn)
print('month_start =', month_start)
print('month_end =', month_end)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
with imp:
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_merchants')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_acq')
    imp.execute('invalidate metadata ods_alpha.scd1_trx')
print('Impala initialized')

## 1) Полная выгрузка merchants по ИНН

In [ ]:
sql_merchants_by_inn = f"""
with p as (
    select '{target_inn}' as target_inn
),
cmp as (
    select
        c.n_cmp,
        cast(c.c_inn as string) as c_inn_raw,
        case
          when length(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')) = 9
            then lpad(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', ''), 10, '0')
          when length(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')) = 11
            then lpad(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', ''), 12, '0')
          else regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')
        end as inn_norm,
        cast(c.c_cmp_name as string) as c_cmp_name,
        c.ods_deleted_flg as cmp_deleted_flg
    from ods_alpha.scd1_companies c
),
cmp_target as (
    select *
    from cmp
    where inn_norm = (select target_inn from p)
),
agr as (
    select
        a.n_cmp_client,
        cast(a.n_agr as string) as n_agr,
        cast(a.abs_agr_id as string) as abs_agr_id,
        cast(a.c_agr_number as string) as c_agr_number,
        cast(a.acq_class as string) as acq_class,
        cast(a.d_valid_from as date) as d_valid_from,
        cast(a.d_valid_to as date) as d_valid_to,
        a.ods_deleted_flg as agr_deleted_flg
    from ods_alpha.scd1_agreements a
)
select
    ct.inn_norm,
    ct.c_inn_raw,
    ct.n_cmp,
    ct.c_cmp_name,
    ct.cmp_deleted_flg,
    a.n_agr,
    a.abs_agr_id,
    a.c_agr_number,
    a.acq_class,
    a.d_valid_from,
    a.d_valid_to,
    a.agr_deleted_flg,
    m.*
from cmp_target ct
left join agr a
  on a.n_cmp_client = ct.n_cmp
left join ods_alpha.scd1_merchants m
  on m.n_cmp = ct.n_cmp
order by ct.n_cmp, a.n_agr, m.c_nmrc
"""

with imp:
    merchants_full = imp.fetch(sql_merchants_by_inn)

print('rows:', len(merchants_full))
display(merchants_full.head(500))

## 2) Сводка по merchants

In [ ]:
sql_merchants_summary = f"""
with p as (
    select '{target_inn}' as target_inn
),
cmp_target as (
    select c.n_cmp
    from ods_alpha.scd1_companies c
    where case
        when length(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')) = 9
          then lpad(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', ''), 10, '0')
        when length(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')) = 11
          then lpad(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', ''), 12, '0')
        else regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')
      end = (select target_inn from p)
),
m as (
    select m.*
    from ods_alpha.scd1_merchants m
    join cmp_target ct on ct.n_cmp = m.n_cmp
)
select
    count(*) as merchant_rows_total,
    count(distinct cast(m.c_nmrc as string)) as retl_cnt_distinct_c_nmrc,
    sum(case when m.c_nmrc is null then 1 else 0 end) as rows_with_null_c_nmrc,
    sum(case when coalesce(m.ods_deleted_flg, '0') = '1' then 1 else 0 end) as rows_deleted_flg_1
from m
"""

with imp:
    merchants_summary = imp.fetch(sql_merchants_summary)

display(merchants_summary)

## 3) Разрез `c_nmrc` + связь с договорами и транзакциями

В этом блоке строим два вывода:
- `forensic_c_nmrc_summary`: по каждой точке `c_nmrc` — есть ли транзакции по договору в месяце и какая сумма;
- `forensic_c_nmrc_agr_detail`: детализация по `c_nmrc + n_agr`.

In [ ]:
sql_forensic_c_nmrc_summary = f"""
with target as (
    select '{target_inn}' as target_inn
),
cmp_target as (
    select c.n_cmp
    from ods_alpha.scd1_companies c
    where case
        when length(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')) = 9
          then lpad(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', ''), 10, '0')
        when length(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')) = 11
          then lpad(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', ''), 12, '0')
        else regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')
      end = (select target_inn from target)
),
agr as (
    select
        a.n_cmp_client,
        cast(a.n_agr as string) as n_agr,
        cast(a.abs_agr_id as string) as abs_agr_id,
        cast(a.c_agr_number as string) as c_agr_number,
        cast(a.acq_class as string) as acq_class
    from ods_alpha.scd1_agreements a
    join cmp_target ct on ct.n_cmp = a.n_cmp_client
    where a.acq_class = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
),
company_retl as (
    select distinct cast(m.c_nmrc as string) as c_nmrc
    from ods_alpha.scd1_merchants m
    join cmp_target ct on ct.n_cmp = m.n_cmp
    where m.c_nmrc is not null
      and coalesce(m.ods_deleted_flg, '0') <> '1'
),
trx_filtered as (
    select
        t.n_trx,
        cast(t.n_amt_src as double) as n_amt_src,
        cast(t.c_nmrc as string) as c_nmrc
    from ods_alpha.scd1_trx t
    where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
      and t.c_trx_class = 'SA'
      and t.c_trx_type = 'S01'
      and t.c_nter is not null
      and coalesce(t.cf_trx_stat, '') <> 'R'
      and coalesce(t.ods_deleted_flg, '0') <> '1'
),
trx_by_agr as (
    select
        a.n_agr,
        a.abs_agr_id,
        a.c_agr_number,
        tf.c_nmrc,
        count(*) as trx_cnt,
        sum(tf.n_amt_src) as trx_sum
    from agr a
    join ods_alpha.scd1_trx_acq ta
      on ta.n_agr = cast(a.n_agr as bigint)
     and coalesce(ta.ods_deleted_flg, '0') <> '1'
    join trx_filtered tf
      on tf.n_trx = ta.n_trx
    group by a.n_agr, a.abs_agr_id, a.c_agr_number, tf.c_nmrc
),
c_nmrc_agg as (
    select
        cr.c_nmrc,
        count(distinct tba.n_agr) as agr_with_trx_cnt,
        sum(coalesce(tba.trx_cnt, 0)) as trx_cnt,
        sum(coalesce(tba.trx_sum, 0)) as trx_sum
    from company_retl cr
    left join trx_by_agr tba
      on tba.c_nmrc = cr.c_nmrc
    group by cr.c_nmrc
)
select
    c_nmrc,
    agr_with_trx_cnt,
    trx_cnt,
    trx_sum,
    case when agr_with_trx_cnt > 0 then 1 else 0 end as used_in_month_trx
from c_nmrc_agg
order by used_in_month_trx asc, c_nmrc
"""

with imp:
    forensic_c_nmrc_summary = imp.fetch(sql_forensic_c_nmrc_summary)

display(forensic_c_nmrc_summary.head(500))
if len(forensic_c_nmrc_summary) > 0:
    print('company_retl_cnt =', len(forensic_c_nmrc_summary))
    print('used_in_month_trx =', int((forensic_c_nmrc_summary['used_in_month_trx'] == 1).sum()))
    print('not_used_in_month_trx =', int((forensic_c_nmrc_summary['used_in_month_trx'] == 0).sum()))

In [ ]:
sql_forensic_c_nmrc_agr_detail = f"""
with target as (
    select '{target_inn}' as target_inn
),
cmp_target as (
    select c.n_cmp
    from ods_alpha.scd1_companies c
    where case
        when length(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')) = 9
          then lpad(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', ''), 10, '0')
        when length(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')) = 11
          then lpad(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', ''), 12, '0')
        else regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')
      end = (select target_inn from target)
),
agr as (
    select
        cast(a.n_agr as string) as n_agr,
        cast(a.abs_agr_id as string) as abs_agr_id,
        cast(a.c_agr_number as string) as c_agr_number
    from ods_alpha.scd1_agreements a
    join cmp_target ct on ct.n_cmp = a.n_cmp_client
    where a.acq_class = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
),
trx_filtered as (
    select
        t.n_trx,
        cast(t.c_nmrc as string) as c_nmrc,
        cast(t.n_amt_src as double) as n_amt_src
    from ods_alpha.scd1_trx t
    where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
      and t.c_trx_class = 'SA'
      and t.c_trx_type = 'S01'
      and t.c_nter is not null
      and coalesce(t.cf_trx_stat, '') <> 'R'
      and coalesce(t.ods_deleted_flg, '0') <> '1'
)
select
    a.n_agr,
    a.abs_agr_id,
    a.c_agr_number,
    tf.c_nmrc,
    count(*) as trx_cnt,
    sum(tf.n_amt_src) as trx_sum
from agr a
join ods_alpha.scd1_trx_acq ta
  on ta.n_agr = cast(a.n_agr as bigint)
 and coalesce(ta.ods_deleted_flg, '0') <> '1'
join trx_filtered tf
  on tf.n_trx = ta.n_trx
where tf.c_nmrc is not null
group by a.n_agr, a.abs_agr_id, a.c_agr_number, tf.c_nmrc
order by a.n_agr, tf.c_nmrc
"""

with imp:
    forensic_c_nmrc_agr_detail = imp.fetch(sql_forensic_c_nmrc_agr_detail)

display(forensic_c_nmrc_agr_detail.head(500))

### 3.1) Количество терминалов по каждому договору (`forensic_terminals_by_agr`)

In [ ]:
sql_forensic_terminals_by_agr = f"""
with target as (
    select '{target_inn}' as target_inn
),
cmp_target as (
    select c.n_cmp
    from ods_alpha.scd1_companies c
    where case
        when length(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')) = 9
          then lpad(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', ''), 10, '0')
        when length(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')) = 11
          then lpad(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', ''), 12, '0')
        else regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')
      end = (select target_inn from target)
),
agr as (
    select
        a.n_cmp_client,
        cast(a.n_agr as string) as n_agr,
        cast(a.abs_agr_id as string) as abs_agr_id,
        cast(a.c_agr_number as string) as c_agr_number
    from ods_alpha.scd1_agreements a
    join cmp_target ct on ct.n_cmp = a.n_cmp_client
    where a.acq_class = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
),
terminals_by_agr as (
    select
        a.n_agr,
        a.abs_agr_id,
        a.c_agr_number,
        count(distinct cast(t.c_nter as string)) as term_cnt
    from agr a
    left join ods_alpha.scd1_merchants m
      on m.n_cmp = a.n_cmp_client
     and coalesce(m.ods_deleted_flg, '0') <> '1'
     and m.c_nmrc is not null
    left join ods_alpha.scd1_pos_terminals t
      on t.c_nmrc = m.c_nmrc
     and coalesce(t.ods_deleted_flg, '0') <> '1'
     and t.c_nter is not null
     and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
     and (t.d_ter_close is null or cast(t.d_ter_close as date) >= cast('{month_start}' as date))
    group by a.n_agr, a.abs_agr_id, a.c_agr_number
)
select
    n_agr,
    abs_agr_id,
    c_agr_number,
    term_cnt
from terminals_by_agr
order by n_agr
"""

with imp:
    forensic_terminals_by_agr = imp.fetch(sql_forensic_terminals_by_agr)

display(forensic_terminals_by_agr.head(500))

In [ ]:
# Объединенный договорный view: trx + retl_with_trx + term_cnt
contract_trx_agg = (
    forensic_c_nmrc_agr_detail
    .groupby(['n_agr', 'abs_agr_id', 'c_agr_number'], as_index=False)
    .agg(
        trx_cnt_total=('trx_cnt', 'sum'),
        trx_sum_total=('trx_sum', 'sum'),
        retl_cnt_with_trx=('c_nmrc', 'nunique')
    )
)

forensic_contract_kpi_view = (
    contract_trx_agg
    .merge(
        forensic_terminals_by_agr,
        on=['n_agr', 'abs_agr_id', 'c_agr_number'],
        how='outer'
    )
    .sort_values(['n_agr', 'c_agr_number'])
    .reset_index(drop=True)
)

for c in ['trx_cnt_total', 'trx_sum_total', 'retl_cnt_with_trx', 'term_cnt']:
    if c in forensic_contract_kpi_view.columns:
        forensic_contract_kpi_view[c] = forensic_contract_kpi_view[c].fillna(0)


display(forensic_contract_kpi_view.head(500))

In [ ]:
# DQ-1: потенциальные дубли терминалов по договору в join-цепочке (agr -> merchants -> terminals)
sql_dq_term_duplicates = f"""
with target as (
    select '{target_inn}' as target_inn
),
cmp_target as (
    select c.n_cmp
    from ods_alpha.scd1_companies c
    where case
        when length(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')) = 9
          then lpad(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', ''), 10, '0')
        when length(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')) = 11
          then lpad(regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', ''), 12, '0')
        else regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '')
      end = (select target_inn from target)
),
agr as (
    select
        a.n_cmp_client,
        cast(a.n_agr as string) as n_agr,
        cast(a.abs_agr_id as string) as abs_agr_id,
        cast(a.c_agr_number as string) as c_agr_number
    from ods_alpha.scd1_agreements a
    join cmp_target ct on ct.n_cmp = a.n_cmp_client
    where a.acq_class = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
),
joined as (
    select
        a.n_agr,
        a.abs_agr_id,
        a.c_agr_number,
        cast(t.c_nter as string) as c_nter,
        cast(m.c_nmrc as string) as c_nmrc
    from agr a
    left join ods_alpha.scd1_merchants m
      on m.n_cmp = a.n_cmp_client
     and coalesce(m.ods_deleted_flg, '0') <> '1'
     and m.c_nmrc is not null
    left join ods_alpha.scd1_pos_terminals t
      on t.c_nmrc = m.c_nmrc
     and coalesce(t.ods_deleted_flg, '0') <> '1'
     and t.c_nter is not null
     and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
     and (t.d_ter_close is null or cast(t.d_ter_close as date) >= cast('{month_start}' as date))
)
select
    n_agr,
    abs_agr_id,
    c_agr_number,
    c_nter,
    count(*) as row_cnt
from joined
where c_nter is not null
group by n_agr, abs_agr_id, c_agr_number, c_nter
having count(*) > 1
order by row_cnt desc, n_agr, c_nter
"""

with imp:
    dq_term_duplicates = imp.fetch(sql_dq_term_duplicates)

display(dq_term_duplicates.head(500))
print('dq_term_duplicates rows =', len(dq_term_duplicates))

In [ ]:
# DQ-2: договоры с trx > 0, но term_cnt = 0
if 'forensic_contract_kpi_view' not in globals():
    raise ValueError('Run the merged contract KPI view cell first.')

dq_trx_positive_term_zero = forensic_contract_kpi_view[
    (forensic_contract_kpi_view['trx_cnt_total'] > 0) &
    (forensic_contract_kpi_view['term_cnt'] == 0)
].copy()

display(dq_trx_positive_term_zero)
print('dq_trx_positive_term_zero rows =', len(dq_trx_positive_term_zero))